In [ ]:
from pybaseball import statcast
import pandas as pd

 Fetch the data and save it to a DataFrame
df = statcast(start_dt="2024-04-01", end_dt="2024-10-02")


In [ ]:
df.head()

In [ ]:
import pandas as pd

 Replace with the correct file path
file_path = r"C:\Users\TrevorWhite\Downloads\report1734638143238.csv"

 Load the CSV file
try:
    data = pd.read_csv(file_path)

     Calculate the maximum length of each column
    max_lengths = data.astype(str).applymap(len).max()

     Print the result
    print("Maximum length of each column:")
    print(max_lengths)
except FileNotFoundError:
    print(f"File not found at the specified path: {file_path}")
except Exception as e:
    print(f"An error occurred: {e}")


In [ ]:
filtered_df = df[
    df['events'].notna() &
    df['launch_angle'].notna() &
    df['launch_speed'].notna() &
    df['hc_x'].notna() &
    df['hc_y'].notna()
]
 Select the specified columns
selected_columns = ['events', 'description', 'launch_speed', 'launch_angle','iso_value']
result_df = filtered_df[selected_columns]

In [ ]:
pd.set_option('display.max_columns', None)

pd.set_option('display.max_rows', None)
# result_df.head(1000)

In [ ]:
slg_mapping = {
        'single': 1,
        'double': 2,
        'triple': 3,
        'home_run': 4
    }

result_df = result_df.copy()   Avoid SettingWithCopyWarning
result_df['slg'] = result_df['events'].map(slg_mapping).fillna(0).astype(int)

In [ ]:
unique_events = result_df['events'].unique()
print("Unique event names in 'events' column:")
for event in unique_events:
    print(event)

In [ ]:
import pandas as pd
import numpy as np
from pybaseball import statcast
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns



 Drop rows with missing values in 'launch_speed', 'launch_angle', or 'slg'
data = result_df.dropna(subset=['launch_speed', 'launch_angle', 'slg'])
data['interaction'] = data['launch_speed'] * data['launch_angle']

  Exploratory Data Analysis (Optional)
 plt.figure(figsize=(10, 6))
 scatter = plt.scatter(data['launch_speed'], data['launch_angle'], c=data['slg'], cmap='viridis', alpha=0.5)
 plt.colorbar(scatter, label='SLG Value')
 plt.xlabel('Launch Speed (mph)')
 plt.ylabel('Launch Angle (degrees)')
 plt.title('Launch Angle vs. Launch Speed colored by SLG')
 plt.grid(True)
 plt.show()

 Features and target variable
X = data[['launch_speed', 'launch_angle', 'interaction']]
y = data['slg']

 Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

 Initialize the XGBoost regressor
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

 Train the model
xgb_model.fit(X_train, y_train)

 Evaluate the model
y_pred = xgb_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test set evaluation metrics:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R² Score: {r2:.4f}")

 Cross-validation
cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)
print(f"\nCross-validation RMSE scores: {cv_rmse}")
print(f"Mean CV RMSE: {cv_rmse.mean():.4f}")

 Feature importance
xgb.plot_importance(xgb_model)
plt.title('Feature Importance')
plt.show()

 Hyperparameter tuning (Optional)
param_grid = {
    'max_depth': [3, 5, 7],
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
}

xgb_regressor = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

grid_search = GridSearchCV(estimator=xgb_regressor,
                           param_grid=param_grid,
                           cv=5,
                           scoring='neg_mean_squared_error',
                           n_jobs=-1)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")

 Evaluate the best model
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
mse_best = mean_squared_error(y_test, y_pred_best)
rmse_best = np.sqrt(mse_best)
mae_best = mean_absolute_error(y_test, y_pred_best)
r2_best = r2_score(y_test, y_pred_best)

print(f"\nBest model evaluation metrics:")
print(f"RMSE: {rmse_best:.4f}")
print(f"MAE: {mae_best:.4f}")
print(f"R² Score: {r2_best:.4f}")


In [ ]:
best_model.save_model('xSLG_model.json')

In [ ]:
import pandas as pd
import numpy as np
from pybaseball import statcast
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns


data = result_df.dropna(subset=['launch_speed', 'launch_angle', 'slg'])

 Create interaction term
data['interaction'] = data['launch_speed'] * data['launch_angle']

 Split the data
X = data[['launch_speed', 'launch_angle', 'interaction']]
y = data['slg']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

 Train the model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

 Evaluate the model
y_pred = lr_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test set evaluation metrics:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R² Score: {r2:.4f}")

 Cross-validation
cv_scores = cross_val_score(lr_model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)
print(f"\nCross-validation RMSE scores: {cv_rmse}")
print(f"Mean CV RMSE: {cv_rmse.mean():.4f}")

 Analyze coefficients
coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_model.coef_
})
intercept = lr_model.intercept_

print("\nModel Coefficients:")
print(coefficients)
print(f"\nIntercept: {intercept:.4f}")

 Optional: Plot observed vs predicted values
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel('Actual SLG')
plt.ylabel('Predicted SLG')
plt.title('Actual vs Predicted SLG')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2)
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import numpy as np

def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Feature engineering for a baseball dataset with columns:
      - relspeed      : pitch velocity
      - spinrate      : spin rate
      - extension     : release extension
      - relheight     : release height
      - relside       : release side (+ => typically R, - => L)
      - horzbreak     : horizontal pitch break
      - vertbreak     : vertical pitch break
      - autopitchtype : pitch type (e.g., "Four-Seam", "Sinker", etc.)
      - pitcher       : pitcher identifier
      - year          : the pitch's year
      ... other columns as needed

    Steps:
      1) Determine pitcher handedness from average 'relside' (R if > 0, else L).
      2) Rename columns to standard references (start_speed, ax, az, etc.).
      3) Mirror horizontal release & break for left-handed pitchers.
      4) From fastball types ["Four-Seam","Sinker"], find the most-used fastball
         per (pitcher, year). If there's a tie, pick the one with the highest
         average speed.
      5) Merge those metrics back & compute diffs:
         - speed_diff = start_speed - avg_fastball_speed
         - az_diff    = az - avg_fastball_az
         - ax_diff    = ax - avg_fastball_ax
    """

     1) DETERMINE PITCHER HANDEDNESS
    print(df.__class__)
    print(df.__module__)
    print(df.head())
        We'll group by 'pitcher' and average 'relside'.
    df_hand = (
        df.groupby("pitcher", as_index=False)["relside"].mean()
          .rename(columns={"relside": "avg_side"})
    )
    df_hand["pitcher_hand"] = np.where(df_hand["avg_side"] > 0, "R", "L")

     Merge handedness info back
    df = pd.merge(df, df_hand[["pitcher", "pitcher_hand"]], on="pitcher", how="left")

     2) RENAME COLUMNS
    df = df.rename(columns={
        "relspeed":      "start_speed",
        "spinrate":      "spin_rate",    if you want to keep spin rate
        "extension":     "extension",    already descriptive
        "relheight":     "z0",           release height
        "relside":       "x0",           release side
        "ax0":     "ax",           horizontal break
        "az0":     "az",           vertical break
        "autopitchtype": "pitch_type"
    })

     3) MIRROR FOR LEFT-HANDED PITCHERS
        If pitcher_hand == "L", flip ax and x0
    df["ax"] = np.where(df["pitcher_hand"] == "L", -df["ax"], df["ax"])
    df["x0"] = np.where(df["pitcher_hand"] == "L", -df["x0"], df["x0"])

     4) MOST-USED FASTBALL LOGIC
        We'll focus on ["Four-Seam","Sinker"] as "fastball_types".
    fastball_types = ["Four-Seam", "Sinker"]
    df_fb = df[df["pitch_type"].isin(fastball_types)].copy()

     Group by (pitcher, year, pitch_type), compute means & usage count
    df_agg = (
        df_fb.groupby(["pitcher", "pitch_type"], as_index=False)
             .agg(
                 avg_fastball_speed=("start_speed", "mean"),
                 avg_fastball_az=("az", "mean"),
                 avg_fastball_ax=("ax", "mean"),
                 count=("start_speed", "count")
             )
    )

     Sort by usage count, then avg_fastball_speed, descending
    df_agg = df_agg.sort_values(["count", "avg_fastball_speed"], ascending=[False, False])

     Keep only the top row (most-used & fastest) per (pitcher, year)
    df_agg = df_agg.drop_duplicates(subset=["pitcher"], keep="first")

     5) MERGE BACK & COMPUTE DIFFS
    df = pd.merge(
        df,
        df_agg[["pitcher",  "avg_fastball_speed", "avg_fastball_az", "avg_fastball_ax"]],
        on=["pitcher"],
        how="left"
    )

    df["speed_diff"] = df["start_speed"] - df["avg_fastball_speed"]
    df["az_diff"]    = df["az"] - df["avg_fastball_az"]
    df["ax_diff"]    = df["ax"] - df["avg_fastball_ax"]
    df["x0"] = df["x0"] * -1

    return df


In [ ]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\usdcsvs\usd_baseball_TM_master_file.csv",
    usecols=[
        "RelSpeed",
        "RelHeight",
        "RelSide",
        "ax0",
        "az0",
        "AutoPitchType",
        "Pitcher",
        "SpinRate",
        "Extension",
         Any other columns your feature_engineering function needs
    ]
)

df = df.rename(columns={col: col.lower() for col in df.columns})


 3) Apply the feature engineering
df_transformed = feature_engineering(df)

 4) Check the results
print(df_transformed.head())


In [ ]:
import joblib

 Suppose df_transformed is a Pandas DataFrame
 and you've defined your feature list:
features = [
    "start_speed",
    "spin_rate",
    "extension",
    "az",
    "ax",
    "x0",
    "z0",
    "speed_diff",
    "az_diff",
    "ax_diff"
]

 1) Load your trained model
model = joblib.load(r"C:\Users\TrevorWhite\Downloads\lgbm_model_2020_2023.joblib")

 2) Predict using your transformed DataFrame (with Pandas indexing)
predictions = model.predict(df_transformed[features])

 3) Insert predictions as a new column named "target"
df_transformed["target"] = predictions

 4) Inspect results
print(df_transformed.head())


In [ ]:
target_mean_2023 = 0.003621590946415154    example
target_std_2023  = 0.006897066586011802    example


df_transformed["target_zscore"] = (
    (df_transformed["target"] - target_mean_2023) / target_std_2023
)

df_transformed["tj_stuff_plus"] = (
    100 - (df_transformed["target_zscore"] * 10)
)

In [ ]:
df_transformed.head()

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
import seaborn as sns
from datetime import datetime
import math
from matplotlib.patches import Arc
from matplotlib.patches import Ellipse
import joblib
import os
import subprocess
import sys
import lightgbm

try:
    import sklearn
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
    import sklearn

import sklearn


 1) DEFINE HELPER FUNCTIONS

def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Feature engineering for a baseball dataset with columns:
      - relspeed      : pitch velocity
      - spinrate      : spin rate
      - extension     : release extension
      - relheight     : release height
      - relside       : release side (+ => typically R, - => L)
      - ax0           : horizontal pitch break
      - az0           : vertical pitch break
      - autopitchtype : pitch type (e.g., "Four-Seam", "Sinker", etc.)
      - pitcher       : pitcher identifier
      ... other columns as needed
    Steps:
      1) Determine pitcher handedness from average 'relside' (R if > 0, else L).
      2) Rename columns to standard references (start_speed, ax, az, etc.).
      3) Mirror horizontal release & break for left-handed pitchers.
      4) From fastball types ["Four-Seam","Sinker"], find the most-used fastball
         per (pitcher). If there's a tie, pick the one with the highest average speed.
      5) Merge those metrics back & compute diffs:
         - speed_diff = start_speed - avg_fastball_speed
         - az_diff    = az - avg_fastball_az
         - ax_diff    = ax - avg_fastball_ax
      6) Flip x0 sign (df["x0"] = df["x0"] * -1) at the end.
    """
     1) Keep only the columns we need
    needed_cols = [
        "pitcher",
        "relside",
        "relspeed",
        "spinrate",
        "extension",
        "relheight",
        "ax0",
        "az0",
        "autopitchtype",
        "pitchuid"
    ]
    df = df[needed_cols].copy()
    
     1) DETERMINE PITCHER HANDEDNESS
    df_hand = (
        df.groupby("pitcher", as_index=False)["relside"].mean()
          .rename(columns={"relside": "avg_side"})
    )
    df_hand["pitcher_hand"] = np.where(df_hand["avg_side"] > 0, "R", "L")

     Merge handedness info back
    df = pd.merge(df, df_hand[["pitcher", "pitcher_hand"]], on="pitcher", how="left")

     2) RENAME COLUMNS
    df = df.rename(columns={
        "relspeed":      "start_speed",
        "spinrate":      "spin_rate",
        "extension":     "extension",
        "relheight":     "z0",
        "relside":       "x0",
        "ax0":           "ax",         
        "az0":           "az",         
        "autopitchtype": "pitch_type"
    })

     3) MIRROR FOR LEFT-HANDED PITCHERS
    df["ax"] = np.where(df["pitcher_hand"] == "L", -df["ax"], df["ax"])
    
    df["x0"] = np.where(df["pitcher_hand"] == "L", -df["x0"], df["x0"])

     4) MOST-USED FASTBALL LOGIC
    fastball_types = ["Four-Seam", "Sinker"]
    df_fb = df[df["pitch_type"].isin(fastball_types)].copy()

    df_agg = (
        df_fb.groupby(["pitcher", "pitch_type"], as_index=False)
             .agg(
                 avg_fastball_speed=("start_speed", "mean"),
                 avg_fastball_az=("az", "mean"),
                 avg_fastball_ax=("ax", "mean"),
                 count=("start_speed", "count")
             )
    )
    df_agg = df_agg.sort_values(["count", "avg_fastball_speed"], ascending=[False, False])
    df_agg = df_agg.drop_duplicates(subset=["pitcher"], keep="first")

    df = pd.merge(
        df,
        df_agg[["pitcher", "avg_fastball_speed", "avg_fastball_az", "avg_fastball_ax"]],
        on=["pitcher"],
        how="left"
    )

    df["speed_diff"] = df["start_speed"] - df["avg_fastball_speed"]
    df["az_diff"]    = df["az"] - df["avg_fastball_az"]
    df["ax_diff"]    = df["ax"] - df["avg_fastball_ax"]

     5) Flip x0 sign
    df["x0"] = df["x0"] * -1

    return df


def run_model_and_scale(df_for_model: pd.DataFrame) -> pd.DataFrame:
    """
    1) Load the trained model from disk.
    2) Predict using the engineered features.
    3) Add 'target' column.
    4) Apply z-score and tj_stuff_plus using pre-known baseline stats.
    Returns a new df with 'target', 'target_zscore', 'tj_stuff_plus'.
    """

     -- LOAD MODEL
     Get the directory of the currently running .py file
    
    
     Construct the path to your joblib file
    model_path = r"C:\Users\TrevorWhite\Downloads\lgbm_model_2020_2023.joblib"
    
     Load the model
    model = joblib.load(model_path)

     -- DEFINE FEATURES
    features = [
        "start_speed",
        "spin_rate",
        "extension",
        "az",
        "ax",
        "x0",
        "z0",
        "speed_diff",
        "az_diff",
        "ax_diff"
    ]

     -- MAKE PREDICTIONS
    predictions = model.predict(df_for_model[features])
    df_for_model["target"] = predictions

     -- APPLY z-score & stuff-plus scaling
    target_mean_2023 = 0.003621590946415154   
    target_std_2023  = 0.006897066586011802

    df_for_model["target_zscore"] = (
        (df_for_model["target"] - target_mean_2023) / target_std_2023
    )
    df_for_model["tj_stuff_plus"] = (
        100 - (df_for_model["target_zscore"] * 10)
    )

    return df_for_model


 Load the CSV file
file_path = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/usd_baseball_TM_master_file.csv"
df = pd.read_csv(file_path)

df.drop_duplicates(subset=['PitchUID'], inplace=True)

 Standardize column capitalization
df.columns = [col.strip().capitalize() for col in df.columns]

df_for_model = df.copy()

 Rename columns to lowercase for the feature_engineering function
df_for_model.columns = [c.lower() for c in df_for_model.columns]

 Ensure the columns needed by feature_engineering exist
 (RelSpeed, RelHeight, RelSide, ax0, az0, AutoPitchType, Pitcher, SpinRate, Extension)
 If any are missing, you may need to handle that or rename them properly.

 1) FEATURE ENGINEERING
df_for_model = feature_engineering(df_for_model)

 2) RUN MODEL + SCALING
df_for_model = run_model_and_scale(df_for_model)

 OPTIONAL: Merge the new columns (target, tj_stuff_plus) back into the original "df"
 so that you can reference them in your existing plots/tables if desired.
 We'll merge on a unique identifier you have (e.g., Pitchuid), if it exists in both.
 For demonstration, let's assume "pitchuid" (lowercase in df_for_model).


In [ ]:
 ... prior code

print("=== Merge Debugging ===")
if "pitchuid" in df_for_model.columns and "Pitchuid" in df.columns:
    print("Found 'pitchuid' in df_for_model.columns and 'Pitchuid' in df.columns.")
    print(f"df_for_model columns: {df_for_model.columns.tolist()}")
    print(f"df (pre-merge) columns: {df.columns.tolist()}")
    print(f"Number of rows in df_for_model: {len(df_for_model)}")
    print(f"Number of rows in df (pre-merge): {len(df)}")

     We select only the new columns from df_for_model we want to bring back
    merged_cols = ["pitchuid", "target", "target_zscore", "tj_stuff_plus"]
    print(f"Merging these columns from df_for_model into df: {merged_cols}")

    df = pd.merge(
        df,
        df_for_model[merged_cols],
        left_on="Pitchuid",
        right_on="pitchuid",
        how="left"
    )

    print("Merge complete.")
    print(f"df (post-merge) columns: {df.columns.tolist()}")
    print(f"Number of rows in df (post-merge): {len(df)}")

     Drop duplicate 'pitchuid' column from df if present
    print("Dropping 'pitchuid' column from df if present...")
    df.drop(columns=["pitchuid"], inplace=True, errors="ignore")
    print(f"df columns (after drop): {df.columns.tolist()}")

else:
    print("Either 'pitchuid' is missing in df_for_model.columns or 'Pitchuid' is missing in df.columns. No merge performed.")
print("=== End Merge Debugging ===")


In [ ]:
df_for_model.head()

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
import seaborn as sns
from datetime import datetime
import math
from matplotlib.patches import Arc
from matplotlib.patches import Ellipse
import joblib
import os
import subprocess
import sys
import lightgbm

try:
    import sklearn
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
    import sklearn

import sklearn


 1) DEFINE HELPER FUNCTIONS

def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Feature engineering for a baseball dataset with columns:
      - relspeed      : pitch velocity
      - spinrate      : spin rate
      - extension     : release extension
      - relheight     : release height
      - relside       : release side (+ => typically R, - => L)
      - ax0           : horizontal pitch break
      - az0           : vertical pitch break
      - autopitchtype : pitch type (e.g., "Four-Seam", "Sinker", etc.)
      - pitcher       : pitcher identifier
      ... other columns as needed
    Steps:
      1) Determine pitcher handedness from average 'relside' (R if > 0, else L).
      2) Rename columns to standard references (start_speed, ax, az, etc.).
      3) Mirror horizontal release & break for left-handed pitchers.
      4) From fastball types ["Four-Seam","Sinker"], find the most-used fastball
         per (pitcher). If there's a tie, pick the one with the highest average speed.
      5) Merge those metrics back & compute diffs:
         - speed_diff = start_speed - avg_fastball_speed
         - az_diff    = az - avg_fastball_az
         - ax_diff    = ax - avg_fastball_ax
      6) Flip x0 sign (df["x0"] = df["x0"] * -1) at the end.
    """
     1) Keep only the columns we need
    needed_cols = [
        "pitcher",
        "relside",
        "relspeed",
        "spinrate",
        "extension",
        "relheight",
        "ax0",
        "az0",
        "autopitchtype",
        "pitchuid"
    ]
    df = df[needed_cols].copy()
    
     1) DETERMINE PITCHER HANDEDNESS
    df_hand = (
        df.groupby("pitcher", as_index=False)["relside"].mean()
          .rename(columns={"relside": "avg_side"})
    )
    df_hand["pitcher_hand"] = np.where(df_hand["avg_side"] > 0, "R", "L")

     Merge handedness info back
    df = pd.merge(df, df_hand[["pitcher", "pitcher_hand"]], on="pitcher", how="left")

     2) RENAME COLUMNS
    df = df.rename(columns={
        "relspeed":      "start_speed",
        "spinrate":      "spin_rate",
        "extension":     "extension",
        "relheight":     "z0",
        "relside":       "x0",
        "ax0":           "ax",         
        "az0":           "az",         
        "autopitchtype": "pitch_type"
    })

     3) MIRROR FOR LEFT-HANDED PITCHERS
    df["ax"] = np.where(df["pitcher_hand"] == "L", -df["ax"], df["ax"])
    print(df["x0"].iloc[0])
    df["x0"] = np.where(df["pitcher_hand"] == "L", -df["x0"], df["x0"])

     4) MOST-USED FASTBALL LOGIC
    fastball_types = ["Four-Seam", "Sinker"]
    df_fb = df[df["pitch_type"].isin(fastball_types)].copy()

    df_agg = (
        df_fb.groupby(["pitcher", "pitch_type"], as_index=False)
             .agg(
                 avg_fastball_speed=("start_speed", "mean"),
                 avg_fastball_az=("az", "mean"),
                 avg_fastball_ax=("ax", "mean"),
                 count=("start_speed", "count")
             )
    )
    df_agg = df_agg.sort_values(["count", "avg_fastball_speed"], ascending=[False, False])
    df_agg = df_agg.drop_duplicates(subset=["pitcher"], keep="first")

    df = pd.merge(
        df,
        df_agg[["pitcher", "avg_fastball_speed", "avg_fastball_az", "avg_fastball_ax"]],
        on=["pitcher"],
        how="left"
    )

    df["speed_diff"] = df["start_speed"] - df["avg_fastball_speed"]
    df["az_diff"]    = df["az"] - df["avg_fastball_az"]
    df["ax_diff"]    = df["ax"] - df["avg_fastball_ax"]

     5) Flip x0 sign
    df["x0"] = df["x0"] * -1

    return df


def run_model_and_scale(df_for_model: pd.DataFrame) -> pd.DataFrame:
    """
    1) Load the trained model from disk.
    2) Predict using the engineered features.
    3) Add 'target' column.
    4) Apply z-score and tj_stuff_plus using pre-known baseline stats.
    Returns a new df with 'target', 'target_zscore', 'tj_stuff_plus'.
    """

     -- LOAD MODEL
     Get the directory of the currently running .py file
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    
     Construct the path to your joblib file
    model_path = r"C:\Users\TrevorWhite\Downloads\lgbm_model_2020_2023.joblib"
    
     Load the model
    model = joblib.load(model_path)

     -- DEFINE FEATURES
    features = [
        "start_speed",
        "spin_rate",
        "extension",
        "az",
        "ax",
        "x0",
        "z0",
        "speed_diff",
        "az_diff",
        "ax_diff"
    ]

     -- MAKE PREDICTIONS
    predictions = model.predict(df_for_model[features])
    df_for_model["target"] = predictions

     -- APPLY z-score & stuff-plus scaling
    target_mean_2023 = 0.003621590946415154   
    target_std_2023  = 0.006897066586011802

    df_for_model["target_zscore"] = (
        (df_for_model["target"] - target_mean_2023) / target_std_2023
    )
    df_for_model["tj_stuff_plus"] = (
        100 - (df_for_model["target_zscore"] * 10)
    )

    return df_for_model


 Load the CSV file
file_path = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/usd_baseball_TM_master_file.csv"
df = pd.read_csv(file_path)

df.drop_duplicates(subset=['PitchUID'], inplace=True)

 Standardize column capitalization
df.columns = [col.strip().capitalize() for col in df.columns]

player_overall_avg_relheight = (
    df.groupby('Pitcher')['Relheight']
    .mean()
    .reset_index()
    .rename(columns={'Relheight': 'player_overall_avg_relheight'})
)

 -------------------------------
 3. Calculate Each Player's Average Relheight per Date (Excluding "Bunnell" and "Jack")
 -------------------------------
 Define pitchers to exclude
excluded_pitchers = ["Bunnell, Jack"]

 Filter out excluded pitchers
df_non_excluded = df[~df['Pitcher'].isin(excluded_pitchers)].copy()

 Calculate average Relheight per player per date
player_date_avg_relheight = (
    df_non_excluded.groupby(['Pitcher', 'Date'])['Relheight']
    .mean()
    .reset_index()
    .rename(columns={'Relheight': 'player_date_avg_relheight'})
)

 -------------------------------
 4. Compute Difference Between Overall and Daily Averages
 -------------------------------
 Merge overall averages with daily averages
player_diff = pd.merge(
    player_date_avg_relheight,
    player_overall_avg_relheight,
    on='Pitcher',
    how='left'
)

 Calculate the difference
player_diff['diff_relheight'] = player_diff['player_overall_avg_relheight'] - player_diff['player_date_avg_relheight']

 -------------------------------
 5. Calculate Average Difference per Date
 -------------------------------
avg_diff_per_date = (
    player_diff.groupby('Date')['diff_relheight']
    .mean()
    .reset_index()
    .rename(columns={'diff_relheight': 'avg_diff_relheight'})
)

 -------------------------------
 6. Merge Average Difference Back to Main DataFrame
 -------------------------------
df = pd.merge(df, avg_diff_per_date, on='Date', how='left')

 -------------------------------
 7. Rename Original 'Relheight' and Create Scaled 'relheight'
 -------------------------------
 Rename the original 'Relheight' to 'relheight_uncleaned'
df.rename(columns={'Relheight': 'relheight_uncleaned'}, inplace=True)

df['avg_diff_relheight'] = df['avg_diff_relheight'].fillna(0)

 Create the new scaled 'relheight' by subtracting the average difference
df['Relheight'] = df['relheight_uncleaned'] + df['avg_diff_relheight']

 -------------------------------
 8. Handle Missing Values (Optional)

df['Relheight'] = df['Relheight'].fillna(df['relheight_uncleaned'])


df_for_model = df.copy()

 Rename columns to lowercase for the feature_engineering function
df_for_model.columns = [c.lower() for c in df_for_model.columns]



 Ensure the columns needed by feature_engineering exist
 (RelSpeed, RelHeight, RelSide, ax0, az0, AutoPitchType, Pitcher, SpinRate, Extension)
 If any are missing, you may need to handle that or rename them properly.

 1) FEATURE ENGINEERING
df_for_model = feature_engineering(df_for_model)

 2) RUN MODEL + SCALING
df_for_model = run_model_and_scale(df_for_model)

 OPTIONAL: Merge the new columns (target, tj_stuff_plus) back into the original "df"
 so that you can reference them in your existing plots/tables if desired.
 We'll merge on a unique identifier you have (e.g., Pitchuid), if it exists in both.
 For demonstration, let's assume "pitchuid" (lowercase in df_for_model).
if "pitchuid" in df_for_model.columns and "Pitchuid" in df.columns:
     We select only the new columns from df_for_model we want to bring back
    merged_cols = ["pitchuid", "target", "target_zscore", "tj_stuff_plus"]
    df = pd.merge(
        df, 
        df_for_model[merged_cols], 
        left_on="Pitchuid", right_on="pitchuid", 
        how="left"
    )
     You might drop the duplicate "pitchuid" column from df
    df.drop(columns=["pitchuid"], inplace=True, errors="ignore")
    
 Load arm angle CSV
armangle_path = "https://raw.githubusercontent.com/tdub29/streamlit-app-1/refs/heads/main/armangle_final_fall_usd.csv"
armangle_df = pd.read_csv(armangle_path)

 Merge arm angle data into df on 'Pitcher'
df = df.merge(armangle_df[['Pitcher', 'armangle_prediction']], on='Pitcher', how='left')

df.dropna(subset=['Date'], inplace=True)
df["datetime"] = pd.to_datetime(df["Date"], errors="coerce")
df["Pitchno"] = pd.to_numeric(df["Pitchno"], errors="coerce")

 2) Add 12 hours (to shift from midnight to noon) 
    plus the minutes indicated by 'Time'
df["datetime"] = (
    df["datetime"]
    + pd.to_timedelta(12, unit="h")          shift to noon
    + pd.to_timedelta(df["Pitchno"], unit="m")   add the minutes from noon
)


df['Pitchtype'] = df['Taggedpitchtype'].replace('Undefined', np.nan).fillna(df['Autopitchtype'])
df['Pitchtype'] = df['Pitchtype'].replace(['Four-Seam', 'FourSeamFastBall'], 'Fastball')

df.to_csv(r"C:\Users\TrevorWhite\Downloads\streamlit_2024_fall_data.csv", index=False)

In [ ]:
decision value

In [ ]:
import pandas as pd
from pybaseball import statcast

def pull_statcast_2023_in_chunks():
     Define monthly date ranges for April through October 2023
    date_ranges = [
        ("2023-04-01", "2023-04-30"),
        ("2023-05-01", "2023-05-31"),
        ("2023-06-01", "2023-06-30"),
        ("2023-07-01", "2023-07-31"),
        ("2023-08-01", "2023-08-31"),
        ("2023-09-01", "2023-09-30"),
        ("2023-10-01", "2023-10-01")   end on Oct 1
    ]
    
     Initialize a list to store each chunk of data
    data_list = []
    
     Pull data month by month
    for start_date, end_date in date_ranges:
        print(f"Pulling data from {start_date} to {end_date} ...")
        monthly_data = statcast(start_dt=start_date, end_dt=end_date)
        data_list.append(monthly_data)
    
     Concatenate all monthly chunks into a single DataFrame
    all_data_2023 = pd.concat(data_list, ignore_index=True)
    
    return all_data_2023

 Call the function
mlb_data_2023 = pull_statcast_2023_in_chunks()

 Now mlb_data_2023 should contain April 1, 2023 through October 1, 2023.
print(f"Total rows retrieved: {len(mlb_data_2023)}")


In [ ]:
mlb_data_2023.head()

In [ ]:
import pandas as pd
import numpy as np

 Example dictionary to group raw pitch descriptions into simpler categories
des_dict = {
    'ball': 'ball',
    'blocked_ball': 'ball',
    'pitchout': 'ball',
    'hit_by_pitch': 'hit_by_pitch',
    'called_strike': 'called_strike',
    'foul': 'foul',
    'foul_tip': 'swinging_strike',
    'swinging_strike': 'swinging_strike',
    'swinging_strike_blocked': 'swinging_strike',
    'hit_into_play': 'hit_into_play',
    'bunt_foul_tip': 'swinging_strike',
    'foul_pitchout': 'foul'
}

 Example dictionary for events that occur when the ball is put in play
ev_dict = {
    'single': 'single',
    'double': 'double',
    'triple': 'triple',
    'home_run': 'home_run',
    'field_out': 'field_out',
    'force_out': 'field_out',
    'grounded_into_double_play': 'field_out',
    'fielders_choice': 'field_out',
    'fielders_choice_out': 'field_out',
    'double_play': 'field_out',
    'sac_fly': 'field_out'
}

--- SAMPLE DataFrame ---
df = mlb_data_2023

 1) Map pitch descriptions into a new column 'des_new'
df['des_new'] = df['description'].map(des_dict)

 2) Create a column 'ev_new' for those that have 'hit_into_play' in 'des_new'
df['ev_new'] = df.loc[df['des_new'] == 'hit_into_play', 'events'].map(ev_dict)

 3) Replace 'des_new' = 'hit_into_play' with the actual event from 'ev_new'
df.loc[df['des_new'] == 'hit_into_play', 'des_new'] = df.loc[df['des_new'] == 'hit_into_play', 'ev_new']

 4) Drop any rows where des_new is still NaN (i.e., unmapped)
df = df.dropna(subset=['des_new'])

df[['description', 'events', 'des_new']].head(10)


In [ ]:

 Display the first 10 unique combos
print(len(mlb_data_2023))

In [ ]:
import pandas as pd

 Load datasets
pitch_data = pd.read_csv(r"C:\Users\TrevorWhite\Downloads\mlb_pitch_data_2020_2024.csv")
run_values = pd.read_csv(r"C:\Users\TrevorWhite\Downloads\run_values.csv")

 Define direct mapping for pitch play_descriptions
play_description_mapping = {
    'Foul': 'foul',
    'Ball': 'ball',
    'Swinging Strike': 'swinging_strike',
    'Called Strike': 'called_strike',
    'Foul Tip': 'foul',
    'Ball In Dirt': 'ball',
    'Hit By Pitch': 'hit_by_pitch',
    'Swinging Strike (Blocked)': 'swinging_strike'
}

 Define mapping for "In Play" play_descriptions
in_play_mapping = {
    'Single': 'single',
    'Double': 'double',
    'Triple': 'triple',
    'Home Run': 'home_run'
}

 Remove rows containing "Bunt" in either play_description or Event
pitch_data = pitch_data[~pitch_data['play_description'].str.contains('Bunt', case=False, na=False)]
pitch_data = pitch_data[~pitch_data['event'].str.contains('Bunt', case=False, na=False)]

 Apply direct mapping for play_descriptions
pitch_data['mapped_event'] = pitch_data['play_description'].map(play_description_mapping)

 Handle "In Play" play_descriptions
pitch_data.loc[pitch_data['play_description'].str.contains('In play', case=False, na=False), 'mapped_event'] = (
    pitch_data['event'].map(in_play_mapping).fillna('field_out')
)

 Check for unmapped rows
unmapped_rows = pitch_data[pitch_data['mapped_event'].isna()]
if not unmapped_rows.empty:
    unmapped_rows.head()

  Merge with run values for final mapping
 final_data = pd.merge(
     pitch_data,
     run_values,
     left_on='mapped_event',
     right_on='event',  'balls' strikes
     how='left'
 )

  Save the final dataset
 final_data.to_csv(r"C:\Users\TrevorWhite\Downloads\mapped_pitch_data.csv", index=False)

 print("Mapping complete! Final data saved to 'mapped_pitch_data.csv'")


In [ ]:
print(mlb_data_2023['description'].unique())

In [ ]:
final_data_just_cols = (
    final_data[['play_description', 'mapped_event', 'delta_run_exp', 'balls', 'strikes']]
    .drop_duplicates()
    .sort_values(by='delta_run_exp', ascending=False)
)

In [ ]:
final_data_just_cols.head(1000)

In [ ]:
pitch_data.head()

In [ ]:
 Ensure 'balls' and 'strikes' exist in both datasets
if 'balls' not in pitch_data.columns or 'strikes' not in pitch_data.columns:
    raise KeyError("Columns 'balls' and 'strikes' must exist in pitch_data.")

if 'balls' not in run_values.columns or 'strikes' not in run_values.columns:
    raise KeyError("Columns 'balls' and 'strikes' must exist in run_values.")

 Merge with run values using mapped_event, balls, and strikes
final_data = pd.merge(
    pitch_data,
    run_values,
    left_on=['mapped_event', 'balls', 'strikes'],
    right_on=['event', 'balls', 'strikes'],
    how='left'
)

 Verify the merge
final_data.head()

  Save the final dataset
 final_data.to_csv(r"C:\Users\TrevorWhite\Downloads\mapped_pitch_data.csv", index=False)
 print("Final data saved to 'mapped_pitch_data.csv'")


In [ ]:
import math
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt


df_model = final_data

df_model = df_model.rename(columns={
    'delta_run_exp_y': 'delta_run_exp',
    'px': 'Platelocside',
    'pz': 'Platelocheight'
})



 1) Define features, target
 Define features after renaming
features = ['Platelocside', 'Platelocheight', 'strikes', 'balls']
target = 'delta_run_exp'





 2) Subset the data
df_model_no_swing = df_model[df_model['is_swing'] != True].dropna(subset=features + [target])
df_model_swing = df_model[df_model['is_swing'] == True].dropna(subset=features + [target])




 3) Train the "No Swing" model

X_no_swing = df_model_no_swing[features]
y_no_swing = df_model_no_swing[target]

X_train_ns, X_test_ns, y_train_ns, y_test_ns = train_test_split(
    X_no_swing, y_no_swing, test_size=0.2, random_state=42
)

dtrain_ns = xgb.DMatrix(X_train_ns, label=y_train_ns)
dtest_ns = xgb.DMatrix(X_test_ns, label=y_test_ns)

params = {
    'objective': 'reg:squarederror',
    'max_depth': 5,
    'learning_rate': 0.1,
    'eval_metric': 'rmse'
}
num_rounds = 100
model_no_swing = xgb.train(params, dtrain_ns, num_rounds)

 Evaluate
pred_no_swing = model_no_swing.predict(dtest_ns)
rmse_no_swing = math.sqrt(mean_squared_error(y_test_ns, pred_no_swing))
print(f"No Swing Model RMSE: {rmse_no_swing:.4f}")

 Feature importance
fi_no_swing = model_no_swing.get_score(importance_type='weight')
print("\nFeature Importance - No Swing:")
for feat, score in sorted(fi_no_swing.items(), key=lambda x: x[1], reverse=True):
    print(f"{feat}: {score}")


 4) Train the "Swing" model

X_swing = df_model_swing[features]
y_swing = df_model_swing[target]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_swing, y_swing, test_size=0.2, random_state=42
)

dtrain_s = xgb.DMatrix(X_train_s, label=y_train_s)
dtest_s = xgb.DMatrix(X_test_s, label=y_test_s)

model_swing = xgb.train(params, dtrain_s, num_rounds)

 Evaluate
pred_swing = model_swing.predict(dtest_s)
rmse_swing = math.sqrt(mean_squared_error(y_test_s, pred_swing))
print(f"\nSwing Model RMSE: {rmse_swing:.4f}")

 Feature importance
fi_swing = model_swing.get_score(importance_type='weight')
print("\nFeature Importance - Swing:")
for feat, score in sorted(fi_swing.items(), key=lambda x: x[1], reverse=True):
    print(f"{feat}: {score}")


 5) Save both models

model_no_swing.save_model(r"C:\Users\TrevorWhite\Downloads\model_no_swing.json")
model_swing.save_model(r"C:\Users\TrevorWhite\Downloads\model_swing.json")

print("\nModels saved to your Downloads folder.")


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb

 -------------------------------------------------
 A1) Split big dataset into No Swing vs. Swing
 -------------------------------------------------
df_model_no_swing = df_model[df_model['is_swing'] != True].dropna(subset=['Platelocside','Platelocheight','strikes','balls'])
df_model_swing = df_model[df_model['is_swing'] == True].dropna(subset=['Platelocside','Platelocheight','strikes','balls'])
 -------------------------------------------------
 A2) Predict using the two models
 -------------------------------------------------
dmat_no_swing = xgb.DMatrix(df_model_no_swing[['Platelocside','Platelocheight','strikes','balls']])
dmat_swing    = xgb.DMatrix(df_model_swing[['Platelocside','Platelocheight','strikes','balls']])

df_model_no_swing['y_pred'] = model_no_swing.predict(dmat_no_swing)
df_model_swing['y_pred']    = model_swing.predict(dmat_swing)

 Combine
df_model_pred = pd.concat([df_model_no_swing, df_model_swing], ignore_index=True)

 -------------------------------------------------
 A3) Group into no-swing, swing, and overall
 -------------------------------------------------
df_no_swing_grouped = df_model_no_swing.groupby(['batter_id','batter_name']).agg(
    mean_pred=('y_pred','mean')
).reset_index()

df_swing_grouped = df_model_swing.groupby(['batter_id','batter_name']).agg(
    mean_pred=('y_pred','mean')
).reset_index()

df_overall_grouped = df_model_pred.groupby(['batter_id','batter_name']).agg(
    mean_pred=('y_pred','mean')
).reset_index()

 (Optional) Filter out minimal pitch thresholds, if you like
 but it's not strictly necessary for deriving the league-wide distribution.
 E.g.: df_no_swing_grouped = df_no_swing_grouped[df_no_swing_grouped['pitches'] >= 500]
 etc.

 -------------------------------------------------
 A4) Extract the distribution stats (mean, std)
 -------------------------------------------------
 No Swing
mu_no_swing  = df_no_swing_grouped['mean_pred'].mean()
std_no_swing = df_no_swing_grouped['mean_pred'].std()
print(f"No Swing: mu={mu_no_swing:.4f}, std={std_no_swing:.4f}")

 Swing
mu_swing  = df_swing_grouped['mean_pred'].mean()
std_swing = df_swing_grouped['mean_pred'].std()
print(f"Swing:    mu={mu_swing:.4f}, std={std_swing:.4f}")

 Overall
mu_overall  = df_overall_grouped['mean_pred'].mean()
std_overall = df_overall_grouped['mean_pred'].std()
print(f"Overall:  mu={mu_overall:.4f}, std={std_overall:.4f}")

 Now we have "league-average" mean_pred and std for no-swing, swing, overall.


In [ ]:
 df_grouped = df_grouped.sort_values('decision_value', ascending=False)
df_overall_grouped.head(10)


In [ ]:
 import pandas as pd

  # Load the CSV file
file_path = r"C:\Users\TrevorWhite\Downloads\2024_aa_movement.csv"
mvdf = pd.read_csv(file_path)

# Set last 3 chars of 'Player' as 'Hand'
mvdf['Hand'] = mvdf['Player'].astype(str).str[-3:]

# Set last 3 chars of '[Glove/Arm-Side Movement (in)]' as 'GLV'
mvdf['GLV'] = mvdf['Glove/Arm-Side Movement (in)'].astype(str).str[-3:]

# Get first 4 chars of '[Glove/Arm-Side Movement (in)]', trim, coerce to numeric
mvdf['GLV_Value'] = pd.to_numeric(mvdf['Glove/Arm-Side Movement (in)'].astype(str).str[:4].str.strip(), errors='coerce')

# Make the number negative if GLV == 'GLV'
mvdf['GLV_Value'] = mvdf.apply(lambda row: -row['GLV_Value'] if row['GLV'] == 'GLV' else row['GLV_Value'], axis=1)

mvdf['GLV_Value'] = mvdf.apply(lambda row: -row['GLV_Value'] if row['Hand'] == 'LHP' else row['GLV_Value'], axis=1)



In [ ]:
# import pandas as pd

# # Load the CSV file
# file_path = r"C:\Users\TrevorWhite\Downloads\2024_aa_movement.csv"
# mvdf = pd.read_csv(file_path)

# # Set last 3 chars of 'Player' as 'Hand'
# mvdf['Hand'] = mvdf['Player'].astype(str).str[-3:]

# # Set last 3 chars of '[Glove/Arm-Side Movement (in)]' as 'GLV'
# mvdf['GLV'] = mvdf['Glove/Arm-Side Movement (in)'].astype(str).str[-3:]

# # Get first 4 chars of '[Glove/Arm-Side Movement (in)]', trim, coerce to numeric
# mvdf['GLV_Value'] = pd.to_numeric(mvdf['Glove/Arm-Side Movement (in)'].astype(str).str[:4].str.strip(), errors='coerce')

# # Make the number negative if GLV == 'GLV'
# mvdf['GLV_Value'] = mvdf['GLV_Value'].where(mvdf['GLV'] != 'GLV', -mvdf['GLV_Value'])

# # Multiply 'hb' (formerly 'GLV_Value') by -1 for 'LHP'
# mvdf['GLV_Value'] = mvdf['GLV_Value'].where(mvdf['Hand'] != 'LHP', -mvdf['GLV_Value'])

# # Rename columns for clarity
# mvdf.rename(columns={'Vertical Movement w/o Gravity (in)': 'ivb', 'GLV_Value': 'hb'}, inplace=True)

# Initialize results list
results = []

# Loop through all mirrored observations
for pitch_type in mvdf['Pitch Type'].unique():
    mvdf_pitch = mvdf[mvdf['Pitch Type'] == pitch_type]
    for n in range(5, 60):
        mvdf_filtered = mvdf_pitch[(mvdf_pitch['Arm Angle'] >= n - 3) & (mvdf_pitch['Arm Angle'] <= n + 3)]
        
        # Mirror hb values for LHP by multiplying by -1
        mirrored_hb = mvdf_filtered['hb'].apply(lambda x: -x if mvdf_filtered['Hand'].iloc[0] == 'LHP' else x)
        
        avg_ivb = mvdf_filtered['ivb'].mean()
        avg_hb = mirrored_hb.mean()
        count = len(mvdf_filtered)
        
        # Append results
        results.append({'Pitch Type': pitch_type, 'n': n, 'avg_ivb': avg_ivb, 'avg_hb': avg_hb, 'count': count})

# Convert results to a DataFrame
avg_results = pd.DataFrame(results)

# Display the resulting DataFrame
print(avg_results.head(111))


In [ ]:
import os

# Define the path to the Downloads folder
# downloads_path = os.path.join(os.path.expanduser("~"), "Downloads")

# # Define the filename
# file_name = "avg_results.csv"

# Combine path and filename
file_path = r"C:\Users\TrevorWhite\Downloads\arm_angle_avg_movement.csv"

# Save the DataFrame to a CSV file
avg_results.to_csv(file_path, index=False)

print(f"File saved to {file_path}")


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde

 Path to your training data
data_path = r"C:\Users\TrevorWhite\Downloads\batted_ball_df.parquet"

 Load the training dataset
df = pd.read_parquet(data_path)

 Filter the data to include only valid batted ball events
df = df.loc[
    (df['spray_deg'] >= 0) & (df['spray_deg'] <= 90) &
    (df['launch_angle'] >= -30) & (df['launch_angle'] <= 60)
]



 Extract league-wide spray and launch angles for the earliest year
earliest_year_data = df.loc[df['game_year'] < 2022]
x_loc_league = earliest_year_data['spray_deg']
y_loc_league = earliest_year_data['launch_angle']

 Prepare data for KDE
values_league = np.vstack([x_loc_league, y_loc_league])

 Fit KDE for the league
kernel_league = gaussian_kde(values_league)

 Define grid for evaluation
X, Y = np.mgrid[0:90:91j, -30:60:91j]
positions = np.vstack([X.ravel(), Y.ravel()])

 Evaluate KDE and normalize
f_league = np.reshape(kernel_league(positions).T, X.shape)
f_league = f_league * (100 / f_league.sum())   Normalize to sum to 100

 Save the baseline KDE and grid for reuse
np.save(r"C:\Users\TrevorWhite\league_kde_earliest.npy", f_league)
np.save(r"C:\Users\TrevorWhite\grid_x.npy", X)
np.save(r"C:\Users\TrevorWhite\grid_y.npy", Y)

print("League baseline for earliest year saved successfully.")
